# 04 — Explorando o grafo: encontrando a fraude

Este notebook adapta o [guia oficial de detecção de fraude do
Neo4j](https://github.com/neo4j-graph-examples/fraud-detection) para o grafo que
você acabou de carregar. Nossa hipótese (a mesma do guia original, adaptada ao
cenário de Pix):

1. **Fraude de primeira parte**: clientes que compartilham RG/e-mail/telefone
   provavelmente são identidades forjadas da mesma pessoa/grupo.
2. **Contas-laranja**: clientes que recebem Pix repetidos e incomuns de um
   fraudador suspeito provavelmente estão ajudando a lavar o dinheiro.

Vamos usar só o que já está no grafo. As propriedades de gabarito (carregadas no
notebook 02) só entram em cena na última seção, para medir o que foi descoberto
contra o que foi injetado de propósito no notebook 01.

In [ ]:
!pip install -q neo4j-rust-ext python-dotenv

## Credenciais

Este notebook busca as credenciais em três lugares, na ordem:

1. **Secrets do Colab** — o ícone de chave 🔑 na barra lateral esquerda.
2. **Variáveis de ambiente** — incluindo um arquivo `.env` na pasta do projeto
   (copie o `.env.example` e preencha). É o caminho para quem roda localmente.
3. **Pergunta na tela** — se não achou nas opções acima, pergunta aqui mesmo.

> ⚠️ **No Colab, cada notebook precisa de permissão para cada secret.** Ter criado
> o secret na sua conta não basta: abra o painel 🔑 e ative a chave
> **"Acesso ao notebook"** (*Notebook access*) para **este** notebook. Sem isso o
> secret é ignorado silenciosamente e o notebook volta a perguntar na tela.
>
> Se os secrets estiverem configurados (e liberados), a célula abaixo não pergunta
> nada — apenas conecta.

Além de poupar digitação, as duas primeiras opções evitam que a URI da sua
instância fique gravada na saída da célula caso você comite o notebook.

In [ ]:
import os
from getpass import getpass

try:  # carrega um arquivo .env, se existir
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass  # python-dotenv não instalado, ou não há .env — segue o baile


def credencial(nome, prompt, secreta=False, padrao=None):
    """Busca em: Secrets do Colab > variável de ambiente (.env) > pergunta na tela."""
    try:
        from google.colab import userdata
        if valor := userdata.get(nome):
            return valor
    except ImportError:
        pass  # não estamos no Colab
    except Exception as e:
        # O caso confuso: o secret existe, mas este notebook não tem permissão.
        # Sem este aviso, o notebook só voltaria a perguntar, sem explicar por quê.
        if "NotebookAccess" in type(e).__name__:
            print(f"⚠️  O secret '{nome}' existe, mas este notebook não tem acesso a ele.")
            print(f"    Abra o painel 🔑 e ative 'Acesso ao notebook' para '{nome}'.")

    if valor := os.environ.get(nome):
        return valor

    return (getpass(prompt) if secreta else input(prompt)).strip() or padrao


NEO4J_URI = credencial("NEO4J_URI", "URI do Neo4j (ex.: neo4j+s://xxxx.databases.neo4j.io): ")
NEO4J_USER = credencial("NEO4J_USERNAME", "Usuário [neo4j]: ", padrao="neo4j")
NEO4J_PASSWORD = credencial("NEO4J_PASSWORD", "Senha: ", secreta=True)
# Atenção: em instâncias AuraDB recentes o banco NÃO se chama "neo4j", e sim o
# próprio instance id (o prefixo da URI). Confira em Aura Console > sua instância,
# ou rode SHOW DATABASES.
NEO4J_DATABASE = credencial("NEO4J_DATABASE", "Nome do banco (geralmente = instance id) [neo4j]: ", padrao="neo4j")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Conectado!")

## 1. Visão geral do grafo

O equivalente ao `CALL apoc.meta.stats()` do guia original. Como cada transação
carrega duas labels (`:Transacao` genérica + o tipo específico), a quebra por
label já mostra `Pix`, `Boleto`, `Compra`, `Deposito` e `Saque` separadamente —
sem precisar de nenhuma consulta especial para isso.

In [ ]:
records, _, _ = driver.execute_query(
    "CALL apoc.meta.stats() YIELD labels, relTypesCount RETURN labels, relTypesCount",
    database_=NEO4J_DATABASE,
)
stats = records[0]
print("Nós por label:", stats["labels"])
print("\nRelacionamentos por tipo:", stats["relTypesCount"])

Uma versão com percentuais, no mesmo estilo do guia original (que faz essa
mesma pergunta sobre `CashIn`/`CashOut`/`Payment`/`Debit`/`Transfer`):

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (t:Transacao)
    WITH count(t) AS total
    UNWIND ['Pix', 'Boleto', 'Compra', 'Deposito', 'Saque'] AS tipo
    CALL apoc.cypher.run('MATCH (t:' + tipo + ') RETURN count(t) AS c', {})
    YIELD value
    RETURN tipo, value.c AS quantidade, round(100.0 * value.c / total, 1) AS percentual
    ORDER BY quantidade DESC
""", database_=NEO4J_DATABASE)
for r in records:
    print(f"{r['tipo']:10s} {r['quantidade']:5d}  ({r['percentual']}%)")

## 2. Módulo 1 — Clientes que compartilham identificadores

Essa é a consulta central do guia original, adaptada para os nossos rótulos. Em
duas linhas de Cypher, respondemos uma pergunta que em SQL exigiria um `SELF JOIN`
triplo com `UNION`: **quais pares de clientes compartilham um RG, e-mail ou
telefone?**

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (c1:Cliente)-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]->(id)<-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]-(c2:Cliente)
    WHERE c1.cpf < c2.cpf
    RETURN c1.cpf AS cliente_1, c2.cpf AS cliente_2, count(*) AS identificadores_em_comum
    ORDER BY identificadores_em_comum DESC
    LIMIT 20
""", database_=NEO4J_DATABASE)
for r in records:
    print(dict(r))

### Compare com o SQL equivalente

Esse é o momento de olhar para o que acabou de acontecer. A consulta acima tem
**uma linha de padrão**:

```cypher
MATCH (c1:Cliente)-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]->(id)<-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]-(c2:Cliente)
```

Na tabela `clientes` do notebook 01, RG, e-mail e telefone eram **colunas**. A
mesma pergunta em SQL exige um self-join por coluna, unidos no fim:

```sql
SELECT cliente_1, cliente_2, COUNT(*) AS em_comum FROM (
    SELECT a.cpf AS cliente_1, b.cpf AS cliente_2
    FROM clientes a JOIN clientes b
      ON a.rg = b.rg              AND a.cpf < b.cpf
    UNION ALL
    SELECT a.cpf, b.cpf
    FROM clientes a JOIN clientes b
      ON a.email = b.email        AND a.cpf < b.cpf
    UNION ALL
    SELECT a.cpf, b.cpf
    FROM clientes a JOIN clientes b
      ON a.telefone = b.telefone  AND a.cpf < b.cpf
) pares
GROUP BY cliente_1, cliente_2
ORDER BY em_comum DESC;
```

Três self-joins, um por coluna. Se o cadastro ganhar um campo novo — endereço,
device id, chave Pix — você escreve um quarto self-join e um quarto `UNION ALL`.
No Cypher, você acrescenta o tipo de relacionamento à lista e a consulta continua
do mesmo tamanho.

E note uma segunda coisa: esse SQL responde apenas **um salto** (pares diretos).
Para achar o *anel inteiro* — "quem se conecta a quem se conecta a quem", em
profundidade desconhecida — seria preciso uma CTE recursiva, e a complexidade
cresce a cada salto. No grafo, é o mesmo `MATCH` de sempre, ou um algoritmo pronto
(é exatamente o que faremos na próxima seção).

**Esse é o argumento central deste material**: não é que o grafo faça algo
impossível em SQL. É que ele torna barato o tipo de pergunta — "quem está
conectado a quem, e a que distância" — que em SQL fica caro o suficiente para
você desistir de perguntar.

Voltando à investigação: nem todo compartilhamento é necessariamente fraude (duas
pessoas de uma mesma família podem legitimamente compartilhar telefone, por
exemplo) — mas é um sinal forte o suficiente para investigar. Vamos materializar
esse padrão como um relacionamento, para poder consultá-lo e agrupá-lo depois sem
repetir o `MATCH` de dois saltos toda vez.

In [ ]:
driver.execute_query("""
    MATCH (c1:Cliente)-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]->(id)<-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]-(c2:Cliente)
    WHERE c1.cpf < c2.cpf
    WITH c1, c2, count(*) AS qtd
    MERGE (c1)-[r:COMPARTILHA_IDENTIFICADOR]->(c2)
    SET r.qtd = qtd
""", database_=NEO4J_DATABASE)

print("Relacionamento COMPARTILHA_IDENTIFICADOR criado.")

## 3. Agrupando em anéis de fraude

Agora queremos ir além dos pares e achar **grupos** de clientes conectados entre
si (o "anel de fraude" completo). Isso é literalmente o problema de *componentes
conectados* — um algoritmo de grafo, não uma consulta de padrão. Vamos rodar
**Weakly Connected Components (WCC)**, que encontra grupos de nós conectados
entre si, direta ou indiretamente.

Vamos rodar via [**Aura Graph
Analytics**](https://neo4j.com/docs/aura/graph-analytics/) (AGA) — sem cobrança no
**AuraDB Free**, através de uma sessão de análise separada (pacote Python
`graphdatascience`).

Um detalhe que vale antecipar: a sessão de análise é **temporária**. Tudo que o
algoritmo descobrir precisa ser **gravado de volta no grafo**, ou desaparece junto
com ela. É o que faremos em cada etapa — e é o que permite, no fim, consultar esses
resultados com um agente de linguagem natural.

In [ ]:
!pip install -q "graphdatascience>=1.15" pandas

### Configurando a sessão de análise

Precisa das **credenciais de API** da Aura e do **instance id** da sua instância.
Como conseguir, se ainda não tem:

1. No [Aura Console](https://console.neo4j.io), clique no seu perfil/organização
   (canto superior direito) e vá em **API credentials** (ou **API Keys**)
2. Clique em **Create API credentials**, dê um nome qualquer (ex.: `workshop`)
3. Copie o **Client ID** e o **Client Secret** — o secret aparece **uma única
   vez**, então salve antes de fechar a janela
4. O **instance id** é o prefixo da sua URI de conexão: em
   `neo4j+s://dbd12345.databases.neo4j.io`, o instance id é `dbd12345`

> ⏱️ **Atenção:** a criação da sessão (`get_or_create`) leva **1 a 2 minutos** na
> primeira vez — é uma máquina de análise sendo provisionada para você. A célula
> parece travada, mas está trabalhando. Rode-a e aproveite para ler a explicação
> da próxima seção enquanto espera.
>
> No tier Free o TTL máximo de uma sessão é **30 minutos** (usar mais que isso dá
> erro), e as sessões não são cobradas.

In [ ]:
from graphdatascience.session import (
    AuraAPICredentials,
    GdsSessions,
    DbmsConnectionInfo,
    AlgorithmCategory,
)
from datetime import timedelta

# Mesma cascata das credenciais do banco: variável de ambiente (.env) >
# Secrets do Colab > pergunta na tela.
AURA_CLIENT_ID = credencial("AURA_CLIENT_ID", "Aura API Client ID: ", secreta=True)
AURA_CLIENT_SECRET = credencial("AURA_CLIENT_SECRET", "Aura API Client Secret: ", secreta=True)
AURA_INSTANCE_ID = credencial("AURA_INSTANCEID", "Aura instance id (ex.: dbd12345, prefixo da URI): ")

sessions = GdsSessions(api_credentials=AuraAPICredentials(AURA_CLIENT_ID, AURA_CLIENT_SECRET))

db_connection = DbmsConnectionInfo(
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    aura_instance_id=AURA_INSTANCE_ID,
)

# a base deste material é pequena (milhares de nós) — o tier menor de memória já basta
memoria = sessions.estimate(
    node_count=5_000,
    relationship_count=20_000,
    algorithm_categories=[AlgorithmCategory.COMMUNITY_DETECTION, AlgorithmCategory.CENTRALITY],
)

gds = sessions.get_or_create(
    session_name="fraude-workshop",
    memory=memoria,
    db_connection=db_connection,
    ttl=timedelta(minutes=30),  # 30 min é o máximo permitido no tier Free
)
gds.v2.verify_session_connectivity()
gds.v2.verify_db_connectivity()
print("Sessão do Aura Graph Analytics pronta.")

Projetamos um grafo remoto com os clientes e o relacionamento
`COMPARTILHA_IDENTIFICADOR` que acabamos de criar, rodamos WCC, e trazemos de
volta os componentes já junto com o `cpf` de cada nó (`db_node_properties`
faz essa ponte entre o id interno do GDS e a propriedade do banco).

In [ ]:
query_projecao_fraude = """
CALL () {
    MATCH (c1:Cliente)
    OPTIONAL MATCH (c1)-[r:COMPARTILHA_IDENTIFICADOR]->(c2:Cliente)
    RETURN c1 AS source, r AS rel, c2 AS target
}
RETURN gds.graph.project.remote(source, target, {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: type(rel)
})
"""

G_fraude, resultado_projecao = gds.v2.graph.project(
    graph_name="fraude",
    query=query_projecao_fraude,
    undirected_relationship_types=["COMPARTILHA_IDENTIFICADOR"],
)
print(f"Grafo 'fraude' projetado: {G_fraude.node_count()} nós, {G_fraude.relationship_count()} relacionamentos")

gds.v2.wcc.mutate(G_fraude, mutate_property="componentId")

df_componentes = gds.v2.graph.node_properties.stream(
    G_fraude, node_properties=["componentId"], db_node_properties=["cpf"]
)
df_componentes.head()

Agora, em Python puro (pandas), filtramos só componentes com 3+ clientes — para
não marcar como suspeito quem só compartilhou telefone com uma pessoa — e gravamos
o resultado de volta no Neo4j com uma escrita Cypher comum (isso não passa pela
sessão do GDS; é só um `SET` no banco de dados).

In [ ]:
grupos = df_componentes.groupby("componentId")["cpf"].apply(list)
aneis_encontrados = [membros for membros in grupos if len(membros) >= 3]

linhas_para_marcar = [
    {"cpf": cid, "grupo": int(grupo), "tamanho": len(membros)}
    for grupo, membros in grupos.items() if len(membros) >= 3
    for cid in membros
]
driver.execute_query(
    """
    UNWIND $rows AS row
    MATCH (c:Cliente {cpf: row.cpf})
    SET c:Suspeito,
        c.grupo_fraude = row.grupo,
        c.tamanho_grupo = row.tamanho
    """,
    rows=linhas_para_marcar,
    database_=NEO4J_DATABASE,
)

clientes_suspeitos = sum(len(a) for a in aneis_encontrados)
print(f"{len(aneis_encontrados)} anéis encontrados, somando {clientes_suspeitos} clientes "
      f"— de {len(df_componentes):,} analisados.\n")

print("Os 5 maiores:")
for anel in sorted(aneis_encontrados, key=len, reverse=True)[:5]:
    print(f"  {len(anel)} clientes: {sorted(anel)}")

Repare na proporção: de 10.000 clientes, o algoritmo isolou pouco mais de uma
centena — e cada grupo é internamente conectado por identificadores em comum. Isso
é uma lista de investigação de tamanho **humano**, extraída de uma base onde
olhar linha por linha seria impossível.

### O que acabou de ser gravado no grafo

Repare que não apenas *calculamos* os grupos — nós **gravamos o resultado de volta**
no banco. Cada cliente de um anel recebeu:

- a label `:Suspeito`
- a propriedade `grupo_fraude` com o id do componente
- a propriedade `tamanho_grupo` com o número de membros do anel

Isso é mais importante do que parece. O algoritmo rodou numa sessão de análise
temporária, que vai ser encerrada no fim deste notebook. Se o resultado ficasse só
no DataFrame, ele morreria com o kernel. Gravado no grafo, ele passa a ser um dado
consultável por qualquer um — inclusive por um agente de linguagem natural, que é
o que faremos no fim.

Uma consulta que era **impossível** antes deste passo:

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (c:Cliente:Suspeito)
    RETURN c.grupo_fraude AS grupo, c.tamanho_grupo AS tamanho,
           collect(c.nome)[0..3] AS alguns_nomes
    ORDER BY tamanho DESC, grupo
    LIMIT 5
""", database_=NEO4J_DATABASE)
for r in records:
    print(dict(r))

## 4. Módulo 2 — Encontrando as contas-laranja

Agora que temos clientes marcados como `:Suspeito`, vamos seguir o dinheiro:
quais clientes **não suspeitos** recebem Pix repetidos e em valores altos vindos
de um `:Suspeito`? Esse é o sinal de conta-laranja. Repare que agora filtramos
pela label `:Pix`, não por uma propriedade — outra vantagem de ter modelado o
tipo da transação como label.

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (fraudador:Cliente:Suspeito)-[:REALIZOU]->(t:Pix)-[:PARA]->(destino:Cliente)
    WHERE NOT destino:Suspeito
    RETURN destino.cpf AS possivel_conta_laranja,
           count(t) AS qtd_pix,
           round(sum(t.valor), 2) AS valor_total_recebido
    ORDER BY valor_total_recebido DESC
    LIMIT 20
""", database_=NEO4J_DATABASE)
possiveis_contas_laranja = [dict(r) for r in records]

for linha in possiveis_contas_laranja:
    print(linha)

Repare no padrão: clientes "normais" recebem um Pix ocasional de valor razoável.
As contas-laranja que injetamos recebem **muitos** Pix, de valores bem mais altos
que a média — e sempre do mesmo pequeno grupo de fraudadores. É esse contraste de
padrão (não uma regra fixa de valor) que o grafo deixa fácil de ver.

## 5. Bônus opcional: PageRank na rede de Pix

O guia original usa **PageRank** para ranquear cada conta-laranja pela
"influência" na rede de transferências suspeitas, em vez de só somar valores.
Vamos rodar e — mais interessante — **comparar honestamente** com o ranking
simples da seção anterior.

Primeiro criamos o relacionamento `TRANSFERIU_PIX_PARA`, agregando o total
transferido de cada fraudador para cada destino. Isso é comum às duas vias
abaixo.

In [ ]:
driver.execute_query("""
    MATCH (f:Cliente:Suspeito)-[:REALIZOU]->(t:Pix)-[:PARA]->(d:Cliente)
    WHERE NOT d:Suspeito
    WITH f, d, sum(t.valor) AS total
    MERGE (f)-[r:TRANSFERIU_PIX_PARA]->(d)
    SET r.valor = total
""", database_=NEO4J_DATABASE)

print("Relacionamento TRANSFERIU_PIX_PARA criado.")

Projetamos a rede de Pix suspeitos e rodamos PageRank ponderado pelo valor.
Reaproveita a sessão `gds` criada na seção 3.

In [ ]:
query_projecao_pix = """
CALL () {
    MATCH (c1:Cliente)
    OPTIONAL MATCH (c1)-[r:TRANSFERIU_PIX_PARA]->(c2:Cliente)
    RETURN c1 AS source, r AS rel, c2 AS target
}
RETURN gds.graph.project.remote(source, target, {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: type(rel),
    relationshipProperties: {valor: rel.valor}
})
"""

G_pix, _ = gds.v2.graph.project(graph_name="rede_pix", query=query_projecao_pix)

gds.v2.page_rank.mutate(G_pix, mutate_property="pagerank", relationship_weight_property="valor")

df_pagerank = gds.v2.graph.node_properties.stream(
    G_pix, node_properties=["pagerank"], db_node_properties=["cpf"]
)
ordenado = df_pagerank.sort_values("pagerank", ascending=False)
ranking_pagerank = ordenado["cpf"].head(15).tolist()
print(ordenado[["cpf", "pagerank"]].head(15).to_string(index=False))

### Cuidado: o PageRank dá score para todo mundo

Antes de gravar, um detalhe que é fácil errar. O PageRank atribui um score **a
todos os nós da projeção**, inclusive aos que não têm nenhuma conexão: eles ficam
com o valor-base do algoritmo (0,15, derivado do `damping factor`). Como projetamos
os 10.000 clientes, 10.000 têm score — e filtrar por `score > 0` não filtra nada.

Confira você mesmo a distribuição:

In [ ]:
print(f"Total de clientes com score:        {len(ordenado):,}")
print(f"Score mínimo (valor-base):         {ordenado['pagerank'].min():.4f}")
print(f"Clientes exatamente no valor-base: {(ordenado['pagerank'] <= 0.1501).sum():,}")

Ou seja: o score só significa algo para quem **de fato recebeu** um Pix de um
suspeito. É esse o conjunto que nos interessa, e é ele que vamos gravar.

Gravamos duas coisas:

- `score_laranja` — o PageRank, apenas nos clientes que receberam de um `:Suspeito`
- a label `:ContaLaranja` — nos que ficaram **acima do 95º percentil** desse grupo

O corte por percentil é o mesmo recurso que o guia oficial do Neo4j usa para
rotular fraudadores: em vez de escolher um número mágico, você deixa a distribuição
dos seus próprios dados decidir onde está a cauda.

In [ ]:
linhas_score = [{"cpf": r.cpf, "score": float(r.pagerank)} for r in ordenado.itertuples()]

# o WHERE é o que importa: só grava em quem recebeu de um suspeito
resumo, _, _ = driver.execute_query(
    """
    UNWIND $rows AS row
    MATCH (c:Cliente {cpf: row.cpf})
    WHERE EXISTS { ()-[:TRANSFERIU_PIX_PARA]->(c) }
    SET c.score_laranja = row.score
    RETURN count(c) AS gravados
    """,
    rows=linhas_score,
    database_=NEO4J_DATABASE,
)
print(f"{resumo[0]['gravados']} clientes receberam score_laranja "
      f"(de {len(linhas_score):,} calculados).")

# corte no 95º percentil, calculado pelo próprio Neo4j sobre esse subconjunto
records, _, _ = driver.execute_query("""
    MATCH (c:Cliente) WHERE c.score_laranja IS NOT NULL
    WITH percentileCont(c.score_laranja, 0.95) AS corte
    MATCH (c:Cliente) WHERE c.score_laranja >= corte
    SET c:ContaLaranja
    RETURN corte, count(c) AS marcadas
""", database_=NEO4J_DATABASE)

r = records[0]
print(f"Corte do 95º percentil: {r['corte']:.4f}")
print(f"{r['marcadas']} clientes marcados como :ContaLaranja")

E agora, a consulta que resume o notebook inteiro — três resultados de algoritmo
combinados, em Cypher trivial, porque tudo já está no grafo:

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (laranja:ContaLaranja)
    OPTIONAL MATCH (fraudador:Suspeito)-[t:TRANSFERIU_PIX_PARA]->(laranja)
    RETURN laranja.cpf AS cpf, laranja.nome AS nome,
           round(laranja.score_laranja, 4) AS score,
           count(DISTINCT fraudador) AS fraudadores_ligados,
           round(sum(t.valor), 2) AS valor_recebido_de_suspeitos
    ORDER BY score DESC
    LIMIT 10
""", database_=NEO4J_DATABASE)
for r in records:
    print(dict(r))

Encerramos a sessão de análise. Note que **os resultados ficam** — eles agora são
propriedades e labels do seu grafo, não estado de um processo.

In [ ]:
gds.v2.graph.drop(G_fraude)
gds.v2.graph.drop(G_pix)
sessions.delete(session_name="fraude-workshop")
print("Sessão do Aura Graph Analytics encerrada. Os resultados seguem no grafo.")

### O algoritmo mais sofisticado ganhou?

Boa pergunta para se fazer sempre. Vamos comparar os dois rankings contra o
gabarito — quantas contas-laranja reais cada método colocou no top 10?

In [ ]:
# top 10 por VALOR recebido — quantos são conta-laranja de verdade?
por_valor, _, _ = driver.execute_query("""
    MATCH (:Cliente:Suspeito)-[:REALIZOU]->(p:Pix)-[:PARA]->(d:Cliente)
    WHERE NOT d:Suspeito
    WITH d, sum(p.valor) AS total
    ORDER BY total DESC
    LIMIT 10
    RETURN sum(CASE WHEN d.gabarito_laranja THEN 1 ELSE 0 END) AS acertos
""", database_=NEO4J_DATABASE)

# top 10 por PAGERANK — mesma pergunta, outro critério
por_pagerank, _, _ = driver.execute_query("""
    MATCH (c:Cliente) WHERE c.score_laranja IS NOT NULL
    WITH c ORDER BY c.score_laranja DESC LIMIT 10
    RETURN sum(CASE WHEN c.gabarito_laranja THEN 1 ELSE 0 END) AS acertos
""", database_=NEO4J_DATABASE)

print(f"Ranking por soma de valor (Cypher simples): {por_valor[0]['acertos']}/10 acertos")
print(f"Ranking por PageRank (GDS):                 {por_pagerank[0]['acertos']}/10 acertos")

Na nossa base, o **Cypher simples costuma empatar ou até ganhar do PageRank**.
Isso não é um defeito do PageRank — é uma lição sobre escolher a ferramenta certa:

- O padrão que injetamos é **concentrado**: cada conta-laranja recebe muito
  dinheiro de **poucos** fraudadores. Somar valores captura isso perfeitamente.
- PageRank premia quem recebe de **muitas fontes distintas e influentes**. Ele
  brilharia num esquema com várias camadas de repasse (fraudador → laranja →
  laranja → saque), onde a "distância" do dinheiro importa mais que o valor
  isolado.

A moral: **rode o algoritmo sofisticado, mas compare com a linha de base simples**
antes de declarar vitória. Um material que só te mostrasse o PageRank ganhando
estaria te ensinando a confiar demais na ferramenta.

## 6. O gabarito — quão perto chegamos?

Hora de medir. As colunas `gabarito_anel` e `gabarito_laranja` que o notebook 01
gerou viraram propriedades dos nós `Cliente` na carga — então a conferência é só
uma consulta de grafo, comparando o que **plantamos** com o que **detectamos**.

Nenhuma consulta de detecção deste notebook olhou para essas propriedades. Elas só
aparecem agora.

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (c:Cliente)
    RETURN
      sum(CASE WHEN c.gabarito_anel IS NOT NULL THEN 1 ELSE 0 END) AS plantados,
      sum(CASE WHEN c:Suspeito THEN 1 ELSE 0 END) AS detectados,
      sum(CASE WHEN c.gabarito_anel IS NOT NULL AND c:Suspeito THEN 1 ELSE 0 END) AS acertos,
      sum(CASE WHEN c.gabarito_anel IS NULL AND c:Suspeito THEN 1 ELSE 0 END) AS falsos_positivos,
      sum(CASE WHEN c.gabarito_anel IS NOT NULL AND NOT c:Suspeito THEN 1 ELSE 0 END) AS escaparam
""", database_=NEO4J_DATABASE)

r = records[0]
print("ANÉIS DE FRAUDE (WCC sobre identificadores compartilhados)")
print(f"  clientes plantados no gabarito : {r['plantados']}")
print(f"  clientes detectados (:Suspeito): {r['detectados']}")
print(f"  acertos                        : {r['acertos']}")
print(f"  falsos positivos               : {r['falsos_positivos']}")
print(f"  escaparam                      : {r['escaparam']}")

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (c:Cliente)
    RETURN
      sum(CASE WHEN c.gabarito_laranja THEN 1 ELSE 0 END) AS plantadas,
      sum(CASE WHEN c:ContaLaranja THEN 1 ELSE 0 END) AS detectadas,
      sum(CASE WHEN c.gabarito_laranja AND c:ContaLaranja THEN 1 ELSE 0 END) AS acertos
""", database_=NEO4J_DATABASE)

r = records[0]
print("CONTAS-LARANJA (PageRank + corte no 95º percentil)")
print(f"  plantadas no gabarito : {r['plantadas']}")
print(f"  marcadas :ContaLaranja: {r['detectadas']}")
print(f"  acertos               : {r['acertos']} de {r['plantadas']}")

Repare no que o gabarito revela sobre o caso mais difícil: uma das
contas-laranja injetadas **também é membro de um anel de fraude**, então ela foi
marcada como `:Suspeito` e o filtro `WHERE NOT destino:Suspeito` a excluiu do
ranking. Nenhum dos dois métodos ia achá-la ali.

Isso não é um bug do material — é como a vida real funciona: as categorias
("fraudador", "laranja") se sobrepõem, e todo filtro que você escreve para achar
um padrão inevitavelmente esconde outro. Vale sempre perguntar *quem o meu filtro
está deixando de fora*.

## Recapitulando

Até aqui, você:

1. Gerou uma base **relacional** (com tabelas, chaves primárias, tabelas
   associativas e uma chave estrangeira polimórfica)
2. Modelou e **carregou** essa base como um grafo no Neo4j, usando `UNWIND` +
   `MERGE`/`CREATE` em lotes, com nós multi-label para representar o tipo da
   transação
3. **Explorou** o grafo com Cypher e com Graph Data Science, encontrando padrões
   de fraude que em SQL exigiriam UNIONs e CTEs recursivas
4. **Comparou** o algoritmo sofisticado com a linha de base simples, em vez de
   assumir que o mais complexo vence
5. **Gravou os resultados de volta no grafo** — e isso é o ponto que muda o que
   vem depois

### O grafo agora sabe coisas que não sabia

A sessão de análise foi encerrada, mas nada do que ela descobriu se perdeu. Seu
grafo ganhou permanentemente:

| O que foi gravado | Onde | Veio de |
|---|---|---|
| `:Suspeito` | label em `Cliente` | WCC |
| `grupo_fraude`, `tamanho_grupo` | propriedades em `Cliente` | WCC |
| `COMPARTILHA_IDENTIFICADOR` | relacionamento | Cypher (seção 2) |
| `TRANSFERIU_PIX_PARA` | relacionamento, com `valor` | Cypher (seção 5) |
| `score_laranja` | propriedade em `Cliente` | PageRank |
| `:ContaLaranja` | label em `Cliente` | PageRank + percentil |

Isso deixou de ser resultado de notebook e passou a ser **dado**. Qualquer pessoa
com acesso ao banco consulta esses achados com um `MATCH` simples, sem precisar
rodar algoritmo nenhum de novo — e é isso que torna possível o passo do
[`aura-agent/`](../aura-agent/README.md), onde um agente responde em linguagem
natural perguntas como *"quais grupos de fraude existem?"*, que dependem
diretamente do `grupo_fraude` que você acabou de gravar.

**Próximo passo:** `05_visualizacao.ipynb` — ver esses mesmos achados desenhados,
onde os anéis e a rede de lavagem viram formas reconhecíveis de olho.

### Para ir além

- [Documentação oficial do Cypher](https://neo4j.com/docs/cypher-manual/current/)
- [Guia oficial de detecção de fraude (Neo4j Graph Examples)](https://github.com/neo4j-graph-examples/fraud-detection) — a versão completa, com mais algoritmos de GDS
- [`neo4j-admin database import`](https://neo4j.com/docs/operations-manual/current/import/) — para cargas muito grandes (milhões+ de linhas), mais rápido que `LOAD CSV`/driver
- [Documentação da Graph Data Science Library](https://neo4j.com/docs/graph-data-science/current/) (referência dos algoritmos)
- [Documentação do Aura Graph Analytics](https://neo4j.com/docs/aura/graph-analytics/) (sessões serverless, inclusive no AuraDB Free)

In [ ]:
driver.close()